# Sales Forecasting Using Random Forest Regression

## Project Overview

This project develops a machine learning model to forecast monthly sales using historical sales data. Time-series features such as lag values, month, quarter, year, and time index are used to train a Random Forest regression model.

The forecast is generated for 2018 and the model is evaluated using MAE, RMSE, and MAPE.
    

: 

## 2. Import Libraries

The required Python libraries are imported for data manipulation, visualization, feature engineering, model training, and evaluation.

## 3. Load and Understand the Dataset

The Superstore sales dataset is loaded from a CSV file. The dataset contains 9,994 records and 21 columns covering order details, customer information, product categories, sales, quantity, discount, and profit.

### Dataset Dimensions

The shape of the dataset is checked to understand the number of rows and columns.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "Sample - Superstore.csv",
    encoding="latin1"
)

df.head()

In [ ]:
df.shape

### Dataset Columns

The column names are inspected to understand the available variables.

In [ ]:
df.columns

### Dataset Information

The dataset information is examined to identify data types, non-null values, and the overall structure of the dataset.

In [ ]:
df.info()

## 4. Data Preprocessing

The `Order Date` column is converted to datetime format so that the sales data can be analyzed chronologically and aggregated by month.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

df['Order Date'].dtype

### Monthly Sales Aggregation

Sales are aggregated by month to create a monthly time-series dataset for forecasting.

In [ ]:
monthly_sales = df.groupby(
    df['Order Date'].dt.to_period('M')
)['Sales'].sum()

monthly_sales.head()

### Convert Monthly Index to Timestamp

The monthly period index is converted to timestamps so that the time series can be used directly for visualization and forecasting.

In [ ]:
monthly_sales.index = monthly_sales.index.to_timestamp()

monthly_sales.head()

### Monthly Sales Trend

The monthly sales trend is visualized to identify changes and patterns in sales over time.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales.values)
plt.title("Monthly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(True)
plt.show()

## 5. Exploratory Data Analysis

The sales data is analyzed at yearly and monthly levels to identify overall trends and recurring seasonal patterns.

### Yearly Sales Analysis

Annual sales are calculated to understand the overall sales trend across the available years.

In [ ]:
yearly_sales = df.groupby(
    df['Order Date'].dt.year
)['Sales'].sum()

yearly_sales

### Monthly Sales Pattern

Sales are grouped by calendar month to identify recurring monthly patterns and seasonality.

In [ ]:
monthly_pattern = df.groupby(
    df['Order Date'].dt.month
)['Sales'].sum()

monthly_pattern


### Seasonal Sales Analysis

Average sales for each calendar month are calculated from the monthly time series to identify seasonal variations in sales.

seasonal_avg = monthly_sales.groupby(
    monthly_sales.index.month
).mean()

seasonal_avg


## 6. Feature Engineering

Time-based and lag features are created from the monthly sales data to help the machine learning model capture trends, seasonality, and historical sales patterns.

### Prepare Forecasting Dataset

In [ ]:
forecast_df = monthly_sales.reset_index()

forecast_df.columns = ['Date', 'Sales']

forecast_df.head()

In [ ]:
### Time-Based Features

Year, month, and quarter features are extracted from the date to capture time-based sales patterns.

In [ ]:
forecast_df['Year'] = forecast_df['Date'].dt.year
forecast_df['Month'] = forecast_df['Date'].dt.month
forecast_df['Quarter'] = forecast_df['Date'].dt.quarter

forecast_df.head()

### Time Index

A sequential time index is created to represent the progression of time in the monthly sales series.

In [ ]:
forecast_df['TimeIndex'] = range(len(forecast_df))

forecast_df.head()

### Lag Features

Lag features are created to provide the model with historical sales information.

- `Lag_1` represents sales from the previous month.
- `Lag_12` represents sales from the same month in the previous year.

In [ ]:
forecast_df['Lag_1'] = forecast_df['Sales'].shift(1)
forecast_df['Lag_12'] = forecast_df['Sales'].shift(12)

forecast_df.head(15)

## 7. Prepare Data for Modeling

Rows containing missing values created by the lag features are removed so that the dataset is ready for machine learning.

In [ ]:
model_df = forecast_df.dropna().copy()

model_df.head()

## 8. Train-Test Split

The data is divided chronologically into training and testing sets. The first 30 monthly observations are used for training, while the final 6 observations are reserved for testing.

In [ ]:
train = model_df[model_df['Date'] < '2017-07-01'].copy()
test = model_df[model_df['Date'] >= '2017-07-01'].copy()

print("Training data:", len(train))
print("Testing data:", len(test))

## 9. Random Forest Regression

The Random Forest Regression algorithm is used to predict monthly sales.

The model uses year, month, quarter, time index, previous-month sales (`Lag_1`), and previous-year sales (`Lag_12`) as input features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

features = ['Year', 'Month', 'Quarter', 'TimeIndex', 'Lag_1', 'Lag_12']

X_train = train[features]
y_train = train['Sales']

X_test = test[features]
y_test = test['Sales']

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

### Train the Random Forest Model

A Random Forest Regressor with 200 trees is trained using the selected features and training data.

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

## 10. Model Evaluation

The trained Random Forest model is used to predict sales for the test period. The predictions are then compared with the actual sales values using standard regression metrics.

In [ ]:
y_pred = model.predict(X_test)



### Evaluation Metrics

The model is evaluated using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). Lower values indicate better prediction performance.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

### Actual vs Predicted Sales

The actual and predicted sales values for the test period are compared to assess how closely the model follows the observed sales.

In [ ]:
comparison = test[['Date', 'Sales']].copy()
comparison['Predicted_Sales'] = y_pred

comparison

### Actual vs Predicted Sales Visualization

The actual and predicted sales values are plotted to visually assess the model's forecasting performance on the test data.

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(comparison['Date'], comparison['Sales'], marker='o', label='Actual Sales')
plt.plot(comparison['Date'], comparison['Predicted_Sales'], marker='o', label='Predicted Sales')

plt.title("Actual vs Predicted Sales — Test Period")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(True)
plt.show()

### Mean Absolute Percentage Error (MAPE)

MAPE measures the average percentage difference between actual and predicted sales. Lower values indicate better forecasting performance.

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAPE (%):", mape * 100)

## 11. Feature Importance

The Random Forest model's feature importance scores are analyzed to identify which time-series features contribute most to the predictions.

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

importance

### Alternative Random Forest Model

A second Random Forest configuration is trained using 500 trees, a maximum depth of 8, and a minimum of 2 samples per leaf. Its performance is compared with the first model.

In [ ]:
model2 = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42
)

model2.fit(X_train, y_train)

In [ ]:
y_pred2 = model2.predict(X_test)

### Evaluation of the Second Random Forest Model

The second Random Forest model is evaluated using MAE, RMSE, and MAPE and compared with the first model.

In [ ]:
mae2 = mean_absolute_error(y_test, y_pred2)
rmse2 = np.sqrt(mean_squared_error(y_test, y_pred2))
mape2 = mean_absolute_percentage_error(y_test, y_pred2)

print("Model 2 MAE:", mae2)
print("Model 2 RMSE:", rmse2)
print("Model 2 MAPE (%):", mape2 * 100)

### Model Performance Comparison

The evaluation metrics of both Random Forest models are summarized to identify the better-performing model.

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest 1', 'Random Forest 2'],
    'MAE': [mae, mae2],
    'RMSE': [rmse, rmse2],
    'MAPE (%)': [mape * 100, mape2 * 100]
})

results

## 12. Train Final Model

Random Forest 1 is selected as the final model because it achieved lower MAE, RMSE, and MAPE than Random Forest 2.

The selected model is retrained using the complete available modeling dataset before generating the 2018 forecast.

In [ ]:
final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(
    model_df[features],
    model_df['Sales']
)

### Generate 2018 Forecast Dates

Twelve monthly dates are generated for 2018 to produce the monthly sales forecast for the target year.

In [ ]:
future_dates = pd.date_range(
    start='2018-01-01',
    periods=12,
    freq='MS'
)

future_dates

## 13. Generate 2018 Forecast

The future dates for all 12 months of 2018 are created, along with the corresponding year, month, quarter, and time-index features required by the final Random Forest model.

In [ ]:
future_df = pd.DataFrame({
    'Date': future_dates
})

future_df['Year'] = future_df['Date'].dt.year
future_df['Month'] = future_df['Date'].dt.month
future_df['Quarter'] = future_df['Date'].dt.quarter
future_df['TimeIndex'] = range(
    model_df['TimeIndex'].max() + 1,
    model_df['TimeIndex'].max() + 13
)

future_df

In [ ]:
history = list(model_df['Sales'])

In [ ]:
jan_2017_sales = model_df.loc[
    model_df['Date'] == '2017-01-01', 'Sales'
].iloc[0]


In [ ]:
jan_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 1,
    'Quarter': 1,
    'TimeIndex': 48,
    'Lag_1': 83829.3188,
    'Lag_12': 43971.374
}])

jan_prediction = final_model.predict(jan_features)[0]

print("Predicted January 2018 Sales:", jan_prediction)

In [ ]:
feb_2017_sales = model_df.loc[
    model_df['Date'] == '2017-02-01', 'Sales'
].iloc[0]



In [ ]:
feb_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 2,
    'Quarter': 1,
    'TimeIndex': 49,
    'Lag_1': jan_prediction,
    'Lag_12': feb_2017_sales
}])

feb_prediction = final_model.predict(feb_features)[0]

print("Predicted February 2018 Sales:", feb_prediction)


In [ ]:
mar_2017_sales = model_df.loc[
    model_df['Date'] == '2017-03-01', 'Sales'
].iloc[0]



In [ ]:
mar_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 3,
    'Quarter': 1,
    'TimeIndex': 50,
    'Lag_1': feb_prediction,
    'Lag_12': mar_2017_sales
}])

mar_prediction = final_model.predict(mar_features)[0]

print("Predicted March 2018 Sales:", mar_prediction)

In [ ]:
apr_2017_sales = model_df.loc[
    model_df['Date'] == '2017-04-01', 'Sales'
].iloc[0]


In [ ]:
apr_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 4,
    'Quarter': 2,
    'TimeIndex': 51,
    'Lag_1': mar_prediction,
    'Lag_12': apr_2017_sales
}])

apr_prediction = final_model.predict(apr_features)[0]

print("Predicted April 2018 Sales:", apr_prediction)

In [ ]:
may_2017_sales = model_df.loc[
    model_df['Date'] == '2017-05-01', 'Sales'
].iloc[0]


In [ ]:
may_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 5,
    'Quarter': 2,
    'TimeIndex': 52,
    'Lag_1': apr_prediction,
    'Lag_12': may_2017_sales
}])

may_prediction = final_model.predict(may_features)[0]

print("Predicted May 2018 Sales:", may_prediction)

In [ ]:
jun_2017_sales = model_df.loc[
    model_df['Date'] == '2017-06-01', 'Sales'
].iloc[0]


In [ ]:
jun_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 6,
    'Quarter': 2,
    'TimeIndex': 53,
    'Lag_1': may_prediction,
    'Lag_12': jun_2017_sales
}])

jun_prediction = final_model.predict(jun_features)[0]

print("Predicted June 2018 Sales:", jun_prediction)

In [ ]:
jul_2017_sales = model_df.loc[
    model_df['Date'] == '2017-07-01', 'Sales'
].iloc[0]



In [ ]:
jul_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 7,
    'Quarter': 3,
    'TimeIndex': 54,
    'Lag_1': jun_prediction,
    'Lag_12': jul_2017_sales
}])

jul_prediction = final_model.predict(jul_features)[0]

print("Predicted July 2018 Sales:", jul_prediction)

In [ ]:
aug_2017_sales = model_df.loc[
    model_df['Date'] == '2017-08-01', 'Sales'
].iloc[0]


In [ ]:
aug_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 8,
    'Quarter': 3,
    'TimeIndex': 55,
    'Lag_1': jul_prediction,
    'Lag_12': aug_2017_sales
}])

aug_prediction = final_model.predict(aug_features)[0]

print("Predicted August 2018 Sales:", aug_prediction)

In [ ]:
sep_2017_sales = model_df.loc[
    model_df['Date'] == '2017-09-01', 'Sales'
].iloc[0]



In [ ]:
sep_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 9,
    'Quarter': 3,
    'TimeIndex': 56,
    'Lag_1': aug_prediction,
    'Lag_12': sep_2017_sales
}])

sep_prediction = final_model.predict(sep_features)[0]

print("Predicted September 2018 Sales:", sep_prediction)

In [ ]:
oct_2017_sales = model_df.loc[
    model_df['Date'] == '2017-10-01', 'Sales'
].iloc[0]



In [ ]:
oct_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 10,
    'Quarter': 4,
    'TimeIndex': 57,
    'Lag_1': sep_prediction,
    'Lag_12': oct_2017_sales
}])

oct_prediction = final_model.predict(oct_features)[0]

print("Predicted October 2018 Sales:", oct_prediction)

In [ ]:
nov_2017_sales = model_df.loc[
    model_df['Date'] == '2017-11-01', 'Sales'
].iloc[0]



In [ ]:
nov_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 11,
    'Quarter': 4,
    'TimeIndex': 58,
    'Lag_1': oct_prediction,
    'Lag_12': nov_2017_sales
}])

nov_prediction = final_model.predict(nov_features)[0]

print("Predicted November 2018 Sales:", nov_prediction)

In [ ]:
dec_2017_sales = model_df.loc[
    model_df['Date'] == '2017-12-01', 'Sales'
].iloc[0]



In [ ]:
dec_features = pd.DataFrame([{
    'Year': 2018,
    'Month': 12,
    'Quarter': 4,
    'TimeIndex': 59,
    'Lag_1': nov_prediction,
    'Lag_12': dec_2017_sales
}])

dec_prediction = final_model.predict(dec_features)[0]

print("Predicted December 2018 Sales:", dec_prediction)

In [ ]:
forecast_2018 = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Sales': [
        jan_prediction,
        feb_prediction,
        mar_prediction,
        apr_prediction,
        may_prediction,
        jun_prediction,
        jul_prediction,
        aug_prediction,
        sep_prediction,
        oct_prediction,
        nov_prediction,
        dec_prediction
    ]
})

forecast_2018

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    forecast_2018['Date'],
    forecast_2018['Predicted_Sales'],
    marker='o'
)

plt.title("2018 Monthly Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Predicted Sales")
plt.grid(True)
plt.show()

In [ ]:

historical = forecast_df[['Date', 'Sales']].copy()
historical['Type'] = 'Actual'


future = forecast_2018[['Date', 'Predicted_Sales']].copy()
future.rename(columns={'Predicted_Sales': 'Sales'}, inplace=True)
future['Type'] = 'Forecast'


powerbi_data = pd.concat(
    [historical, future],
    ignore_index=True
)

powerbi_data.head()

In [ ]:
powerbi_data.to_csv(
    r"C:\Users\BHOOMIKA\OneDrive\Desktop\FUTURE_ML_01\dataset\powerbi_sales_forecast.csv",
    index=False
)

print("CSV saved successfully!")